# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset details the analysis of protein abundance, modifications, and sequences in human samples using mass spectrometry. The following steps will walk you through loading the dataset, reviewing its structure, extracting specific records, performing exploratory analysis, and basic visualization.

### Dataset Source
The dataset is described with a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the mlcroissant API. This cell retrieves and prints the dataset title and description to confirm successful loading.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

# Print dataset title and description
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview

Explore available Record Sets, their `@id`s, fields, and columns using the mlcroissant metadata. This process helps us understand the structure before extracting data.

- **Record sets (`cr:RecordSet`)** identify major data tables/groups.
- **Fields (`cr:Field`)** describe each variable/column.
- Refer to all elements by their `@id`.

In [ ]:
# Extract record sets and their fields by @id
def get_entities_of_type(metadata, type_uri):
    entities = []
    def walk(obj):
        if isinstance(obj, dict):
            if '@type' in obj and (obj['@type'] == type_uri or (isinstance(obj['@type'], list) and type_uri in obj['@type'])):
                entities.append(obj)
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for v in obj:
                walk(v)
    metadata_dict = dataset.metadata.to_json()
    walk(metadata_dict)
    return entities

# RecordSet and Field types
record_set_type = "cr:RecordSet"
field_type = "cr:Field"

# Find all record sets
record_sets = get_entities_of_type(dataset.metadata.to_json(), record_set_type)
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"- Record Set: {rs.get('@id')} | name: {rs.get('name','N/A')}")
    record_set_ids.append(rs['@id'])
    # Show contained fields (columns)
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            print(f"    - Field/column id: {fid}")
    if 'column' in rs:  # Some Croissant schemas use 'column'
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        for c in columns:
            cid = c['@id'] if isinstance(c, dict) and '@id' in c else str(c)
            print(f"    - Column id: {cid}")
if not record_sets:
    print("No explicit cr:RecordSet entities found in schema. Attempting to infer from distribution files...")
    # For some simple datasets, there's only a single table/file, not declared as a RecordSet
    # Try listing 'distribution' entries
    distributions = dataset.metadata.to_json().get('distribution', [])
    for dist in (distributions if isinstance(distributions, list) else [distributions]):
        print(f"Distribution: {dist.get('@id','(no id)')}")

## 3. Data Extraction

Based on the available record set(s) discovered, extract data from each as a pandas `DataFrame`.
All extraction references the record set by its `@id`.

If the schema lacks explicit `cr:RecordSet` entities but data is present in `distribution`, use those.

In [ ]:
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading data from record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (fields by @id):")
        print(df.columns.tolist())
        display(df.head())
else:
    # Fallback: try using distributions
    distributions = dataset.metadata.to_json().get('distribution', [])
    for dist in (distributions if isinstance(distributions, list) else [distributions]):
        dist_id = dist.get('@id', None)
        print(f"Loading data from distribution: {dist_id}")
        try:
            records = list(dataset.records())  # If only one data table
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            print(f"Loaded {len(df)} records. Columns:")
            print(df.columns.tolist())
            display(df.head())
        except Exception as e:
            print("Could not load records from distribution", dist_id, "-", str(e))
    if not distributions:
        print("No data table found to load.")

## 4. Exploratory Data Analysis (EDA)

Demonstrate typical data processing using column `@id` for all references. For this dataset, let's:
- Select a numeric field (e.g., molecular weight, peptide count, or abundance field)
- Filter records,
- Normalize the column,
- Group results by a category (e.g., protein accession or other class).

**Note:** Field/column `@id`s may be verbose. They are visible in output above.

In [ ]:
# Example: Use first available dataframe
from IPython.display import display
import numpy as np

# Automatically select first table and some numeric field
example_df_key = next(iter(dataframes.keys()))
df = dataframes[example_df_key]

print(f"Analyzing data from: {example_df_key}")

# Guess some numeric columns
numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int] or df[c].apply(lambda x: isinstance(x,(int,float,np.integer,np.floating))).all()]
if not numeric_candidates:
    # Try to coerce likely numeric columns
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except Exception:
            continue
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field @id: {numeric_field_id}")
    # Set arbitrary threshold for filtering (e.g., median)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold} (median, {filtered_df.shape[0]}/{df.shape[0]} records):")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a string/categorical field if available
    potential_group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
    if potential_group_fields:
        group_field = potential_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean {numeric_field_id} by group field @id: {group_field}")
        display(grouped_df.head())
    else:
        print("No obvious group field detected.")
else:
    print("No numeric field found in dataframe.")

## 5. Visualization

Visualize the numeric field's distribution and/or relationship to the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of numeric field (by @id): {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field exists, boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion

In this notebook, you explored the FAIR² dataset describing human protein data from extracellular vesicles processed by mass spectrometry.

- Using the Croissant schema, all data elements were referenced by `@id` for clarity and provenance.
- You reviewed available record sets and their fields, extracted data as pandas DataFrames, and performed simple filtering, normalization, grouping, and plotting operations.

You can now use these data and techniques for more advanced analysis or machine learning.